In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
import pandas as pd

In [0]:
pip install openpyxl

In [0]:
data_hta = spark.read\
.format("csv")\
.option("header",True)\
.load("/Volumes/workspace/hta_sis/raw/HTA_FUAS.csv")

In [0]:
data_diag = spark.read\
    .format("csv")\
    .option("header",True)\
    .load("/Volumes/workspace/hta_sis/raw/HTA_Diagnosticos.csv")

In [0]:
import pandas as pd


ruta_excel = "/Volumes/workspace/hta_sis/raw/6. Catálogos.xlsx"

sheets = pd.read_excel(ruta_excel, sheet_name=None, engine="openpyxl")

def limpiar_columnas(df):
    df.columns = (
        df.columns
        .str.strip()                           
        .str.replace(" ", "_")                
        .str.replace("Á", "A").str.replace("É", "E")
        .str.replace("Í", "I").str.replace("Ó", "O")
        .str.replace("Ú", "U").str.replace("Ñ", "N")
        .str.replace("[^0-9a-zA-Z_]", "", regex=True)  
        .str.lower()                          
    )
    return df


for nombre, df in sheets.items():
    print(f"📄 Procesando hoja: {nombre}...")
    df = limpiar_columnas(df)  
    df_spark = spark.createDataFrame(df) 
    
    destino = f"/Volumes/workspace/hta_sis/bronze/{nombre.lower()}"
    df_spark.write.format("delta").mode("overwrite").save(destino)
    print(f"✅ Guardado en {destino}")

In [0]:
data_medicamentos = spark.read\
    .format("csv")\
    .option("header",True)\
    .load("/Volumes/workspace/hta_sis/raw/HTA_Medicamentos.csv")

In [0]:
data_procedimientos = spark.read\
    .format("csv")\
    .option("header",True)\
    .load("/Volumes/workspace/hta_sis/raw/HTA_Procedimientos.csv")

In [0]:
data_insumos = spark.read\
    .format("csv")\
    .option("header",True)\
    .load("/Volumes/workspace/hta_sis/raw/HTA_Insumos.csv")

In [0]:
merge_final_data = spark.read\
    .format("parquet")\
    .load("/Volumes/workspace/hta_sis/raw/merge_sjm_completo")

In [0]:
data_hta.write.format("delta").mode("overwrite").save("/Volumes/workspace/hta_sis/bronze/fact_hta")
data_diag.write.format("delta").mode("overwrite").save("/Volumes/workspace/hta_sis/bronze/fact_diag")
data_medicamentos.write.format("delta").mode("overwrite").save("/Volumes/workspace/hta_sis/bronze/fact_medicamentos")
data_procedimientos.write.format("delta").mode("overwrite").save("/Volumes/workspace/hta_sis/bronze/fact_procedimientos")
data_insumos.write.format("delta").mode("overwrite").save("/Volumes/workspace/hta_sis/bronze/fact_insumos")

In [0]:
merge_final_data.write.format("delta").mode("overwrite").save("/Volumes/workspace/hta_sis/bronze/fact_merge_final")